# Task 2: Implementation — Review Listing & Predictive Labeling

## 1. Overview

This notebook documents the backend implementation for **Task 2: Review Listing**. The objective is to provide a robust system for handling new user reviews, which includes an automated "Buyer/Non-Buyer" labeling system powered by a predictive ensemble model.

### Core Requirements Addressed:
1. **Review Processing**: Handling raw user input (text, title, rating) and preparing it for classification.
2. **Predictive Labeling**: Integrating three machine learning models into a soft-voting ensemble to predict if a customer "Would Buy" or "Would Not Buy" the item.
3. **Persistence and Retrieval**: Ensuring new reviews are saved and immediately reflected in the product catalog.

---


## 2. Environment Setup and Data Loading

To support the predictive modeling, we load the pre-trained artifacts generated during the research phase. This includes our ensemble of models and the vocabulary/stopwords required for consistent preprocessing.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import pickle
from datetime import datetime

# Add parent directory to path to access core modules
sys.path.append(os.path.abspath('..'))

from core.data_loader import load_all, add_review
from core.classifier import predict_buyer
from core.preprocessing import preprocess_text

print("Loading system components...")
data = load_all()

# Extract models and supporting data
models = data['models']
products_df = data['products_df']
stop_words = data['stopwords']
colloc_dict = models['collocation_dict']
label_encoder = models['label_encoder']

print(f"\nSystem Ready. Loaded {len(models)} model artifacts.")


## 3. Predictive Model Implementation (Task 2.2)

The core of Task 2 is the automated labeling system. Rather than relying on a single model, we implemented a **Soft-Voting Ensemble** consisting of three distinct architectures. This approach improves robustness by averaging the confidence (probabilities) of multiple models that "view" the data differently.

### Ensemble Architecture
| Model Type | Feature Representation | Input Features | Strength |
|---|---|---|---|
| **Random Forest** | GloVe Embeddings (Mean) | Review Text | Lexical signal + Product popularity. |
| **Random Forest** | Bag-of-Words (BoW) | Text + Title | Semantic meaning + Product metadata. |
| **Random Forest** | GloVe + Metadata | Text + Title + Product Meta | Importance-weighted semantics + Metadata. |

**Justification for Soft Voting:**
Unlike hard voting (which only looks at the binary output), soft voting averages the predicted probabilities. This allows high-confidence predictions from one model to appropriately influence the final outcome when other models are uncertain.


In [ ]:
def process_and_label_review(review_data):
    """
    Receives raw review data, applies the predictive ensemble, and returns a labeled review.
    """
    text = review_data['review_text']
    title = review_data['review_title']
    rating = review_data['review_rating']
    product_id = review_data['product_id']

    # Retrieve product metadata
    product = products_df[products_df['product_id'] == product_id].iloc[0]
    product_info = {
        'price': product['price'],
        'avg_product_rating': product['avg_product_rating'],
        'product_rating_count': product.get('product_rating_count', 0),
        'brand_name': product['brand_name'],
    }

    # Execute prediction ensemble
    # predict_buyer handles preprocessing (collocations, stopwords) internally
    prediction = predict_buyer(
        text, title, rating, product_info,
        models['rf_bow_extra'], 
        models['rf_unweighted_extra'], 
        models['rf_weighted_extra'], 
        label_encoder, 
        colloc_dict, 
        stop_words
    )

    # Construct the final review object
    labeled_review = review_data.copy()
    labeled_review['predicted_label'] = prediction['label']
    labeled_review['prediction_confidence'] = prediction['fused_prob']
    
    return labeled_review

print("Review processing logic initialized.")


## 4. Saving and Integration (Task 2.3)

Once a review is labeled, it must be persisted. In this implementation, we allow for a "User Confirmation" step (where users can override a misclassification) before final saving. This ensures data integrity while providing a seamless UX.


In [ ]:
def finalize_review(labeled_review, user_would_buy=True):
    """
    Finalizes the review based on user confirmation (Would Buy?) and saves it to the system.
    """
    # Override prediction if user provides manual confirmation
    labeled_review['is_buyer'] = 1 if user_bought_it else 0
    
    # Add timestamp and ID
    labeled_review['review_date'] = datetime.now().strftime('%d/%m/%Y %H:%M')
    
    # Save to the data store (core.data_loader handles memory/disk persistence)
    add_review(labeled_review)
    
    return labeled_review


## 5. Implementation Demo: End-to-End Review Flow

We demonstrate the full backend flow: from receiving raw input, to automated labeling, to final storage.


In [ ]:
# 1. Mock user input (Review for a product they likely bought)
raw_input = {
    'product_id': 148, # Random product ID from catalog
    'review_title': 'My favorite daily serum!',
    'review_text': 'I have been using this for 3 months now. It makes my skin so smooth and hydrated. Definitely a repeat purchase.',
    'review_rating': 5
}

# 2. Process and Label (Task 2.2)
print(f"Processing review for Product ID: {raw_input['product_id']}...")
labeled = process_and_label_review(raw_input)
print(f"-> Automated Prediction: {labeled['predicted_label']} (Confidence: {labeled['prediction_confidence']:.2%})")

# 3. Finalize and Save (Task 2.3)
saved_review = finalize_review(labeled, user_would_buy=True)
print(f"-> Review successfully saved on {saved_review['review_date']}.")
print("\n--- Final Review Data Structure ---")
import json
print(json.dumps(saved_review, indent=2))


## 6. Conclusion and Justification

The implemented system for Task 2 is built on a data-driven foundation:

- **Evidence-Based Model Choice**: We implemented a Random Forest ensemble because our Milestone 1 research identified these as the top-performing architectures. Specifically, the ensemble fuses the three highest-ranking models from our research:

| Rank | Features | Model | F1 (Macro) |
|---|---|---|---|
| 1 | Text+Title+Extra (Unweighted GloVe) | **Random Forest** | **0.6808** |
| 2 | Text+Title+Extra (Weighted GloVe) | **Random Forest** | **0.6798** |
| 3 | Text+Title+Extra (Bag-of-Words) | **Random Forest** | **0.6711** |

- **Technical Implementation**: By orchestrating these three models in a **Soft-Voting Ensemble**, we integrate raw text signals with product-level metadata. This maximizes prediction accuracy by ensuring the system is not skewed by any single feature type (sparse BoW vs. dense Embeddings).

- **System Orchestration**: The review saving logic is decoupled from the UI, acting as a system catalyst. Once a review is saved via `add_review`, it immediately updates downstream components like the Sentiment Dashboard and Recommendation engine, ensuring a unified and consistent user experience.
